In [ ]:
!gcloud auth application-default login

In [1]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()


Loading BokehJS ...

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/20 18:48:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
path_to_release_folder="gs://open-targets-pipeline-runs/szsz/25.06-testrun-1/"


si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

In [3]:
fm=session.spark.read.parquet(path_to_release_folder+"intermediate/l2g_feature_matrix/")

In [4]:
target=session.spark.read.parquet(path_to_release_folder+"output/target/")

In [5]:
target.printSchema()

root
 |-- id: string (nullable = true)
 |-- approvedSymbol: string (nullable = true)
 |-- biotype: string (nullable = true)
 |-- transcriptIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- canonicalTranscript: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: string (nullable = true)
 |-- canonicalExons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- genomicLocation: struct (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: integer (nullable = true)
 |-- alternativeGenes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- approvedName: string (nullable = true)
 |-- go: array (nullable = true)
 |    |-- element: struct (containsNull = tru

In [12]:
target_pc=target.select("id","biotype").withColumnRenamed("id","geneId").cache()
target_pc.count()

25/05/20 19:00:32 WARN CacheManager: Asked to cache already cached data.


78726

In [13]:
target.groupBy("biotype").count().show()

+--------------------+-----+
|             biotype|count|
+--------------------+-----+
|transcribed_unita...|  185|
|                rRNA|   47|
|              lncRNA|34882|
|               snRNA| 1901|
|transcribed_proce...| 1155|
|           IG_C_gene|   14|
|  unitary_pseudogene|   81|
|           TR_V_gene|  112|
|      protein_coding|20130|
|processed_pseudogene| 9486|
|               miRNA| 1879|
|           TR_J_gene|   79|
|     IG_J_pseudogene|    3|
|transcribed_unpro...| 1596|
|              snoRNA|  942|
|     rRNA_pseudogene|  497|
|     IG_V_pseudogene|  187|
|           IG_V_gene|  147|
|unprocessed_pseud...| 1953|
|            misc_RNA| 2207|
+--------------------+-----+
only showing top 20 rows



In [14]:
target_pc = target_pc.withColumn(
    "true_protein_coding",
    f.when(f.col("biotype") == "protein_coding", 1).otherwise(0)
).drop("biotype").cache()
target_pc.count()

78726

In [15]:
target_pc.show(1)

+---------------+-------------------+
|         geneId|true_protein_coding|
+---------------+-------------------+
|ENSG00000096384|                  1|
+---------------+-------------------+
only showing top 1 row



In [16]:
fm_f=fm.select("geneId","isProteinCoding").join(target_pc,on="geneId",how="inner").cache()
fm_f.count()

32612347

In [17]:
fm_f = fm_f.withColumn(
    "corresponding_protein_coding",
    f.when(f.col("isProteinCoding") == f.col("true_protein_coding"), 1).otherwise(0)
).cache()
fm_f.count()

32612347

In [18]:
fm_f.groupBy("corresponding_protein_coding").count().show()

+----------------------------+--------+
|corresponding_protein_coding|   count|
+----------------------------+--------+
|                           1|29003943|
|                           0| 3608404|
+----------------------------+--------+



In [19]:
3608404/32612347

0.11064533319236423

In [21]:
from gentropy.dataset.target_index import TargetIndex
target_index_path=path_to_release_folder+"output/target/"
target_index = TargetIndex.from_parquet(
                session, target_index_path, recursiveFileLookup=True
            )

In [22]:
target_index_df = target_index.df.select("id","biotype").withColumnRenamed("id","geneId")

target_index_df=target_index_df.withColumn(
    "isProteinCoding",
    f.when(f.col("biotype") == "protein_coding", 1).otherwise(0)
).drop("biotype").cache()
target_index_df.count()

78726